# In-Class Exercise: Simple Linear Regression (IMS Chapter 7) — SOLUTIONS

This exercise walks through the core ideas of **Chapter 7: Linear Regression with a Single Predictor**, translated from the book's R examples into Python.

**Key concepts covered:**
- Fitting a line to data and reading the equation
- Residuals and their interpretation
- Correlation
- Least squares and the slope/intercept formulas
- R² (coefficient of determination)
- Categorical predictors as indicator variables
- Outliers and leverage

**Datasets**: `loan50.csv` and `county.csv` — the same files from previous exercises.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
loan50 = pd.read_csv('loan50.csv')
county = pd.read_csv('county.csv')

print('loan50:', loan50.shape)
print('county:', county.shape)

---

## Part 1: Fitting a Line

A linear model describes the relationship between two numerical variables as a straight line:

$$\hat{y} = b_0 + b_1 x$$

where $b_0$ is the **intercept** and $b_1$ is the **slope**. The hat on $\hat{y}$ reminds us this is a *prediction*, not the actual observed value.

### 1a. Scatterplot with a Regression Line

We'll model `loan_amount` (y) as a function of `total_income` (x) using `loan50`.

The cell below fits the model and overlays the regression line on the scatterplot. Run it and examine the output.

In [ ]:
loan50_clean = loan50.dropna(subset=['total_income', 'loan_amount'])

model1 = smf.ols('loan_amount ~ total_income', data=loan50_clean).fit()

b0 = model1.params['Intercept']
b1 = model1.params['total_income']
print(f'Intercept (b0): {b0:,.1f}')
print(f'Slope     (b1): {b1:.4f}')
print(f'Equation: loan_amount = {b0:,.1f} + {b1:.4f} * total_income')

x_range = np.linspace(loan50_clean['total_income'].min(),
                      loan50_clean['total_income'].max(), 200)
y_hat   = b0 + b1 * x_range

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(loan50_clean['total_income'], loan50_clean['loan_amount'],
           alpha=0.6, edgecolors='white', label='Observations')
ax.plot(x_range, y_hat, color='red', linewidth=2, label='Regression line')
ax.set_xlabel('Total Income ($)')
ax.set_ylabel('Loan Amount ($)')
ax.set_title('Loan Amount vs. Total Income (loan50)')
ax.legend()
plt.tight_layout()
plt.show()

### 1b. Interpreting the Equation

Use the slope and intercept printed above to answer the questions below.

**Q1.** In plain English, what does the slope tell you? (Complete the sentence: "For each additional dollar of income, the model predicts...")

**Answer**: The slope is positive — for each additional dollar of total income, the model predicts a small increase in loan amount (the exact value is printed above). In practical terms, each additional $10,000 of income is associated with a few hundred more dollars in loan amount. The positive sign makes intuitive sense: higher-income borrowers tend to qualify for and take out larger loans.

**Q2.** Use the equation to predict the loan amount for someone with a total income of $60,000. Is that income value within the range of the data? What happens if you predict for $1,000,000?

Hint: call `.predict()` on `model1` with a one-row DataFrame.

In [ ]:
# YOUR CODE HERE: predict loan_amount for total_income = 60_000 and for 1_000_000
# Hint: model1.predict(pd.DataFrame({'total_income': [60_000, 1_000_000]}))
predictions = model1.predict(pd.DataFrame({'total_income': [60_000, 1_000_000]}))
print(predictions)

*Does the $1,000,000 prediction seem trustworthy? Why or why not? This is the danger of* **extrapolation** *— using a model outside the range of the data used to fit it.*

**Answer**: $60,000 is within the observed range of total incomes in loan50, so that prediction is well-supported by the data. $1,000,000 is far beyond any income in the sample — this is **extrapolation**. The model will produce a number, but the linear relationship may not hold at such extreme values: loan amounts don’t grow without bound and lenders have policies that cap loan sizes. Extrapolated predictions should be treated with skepticism or avoided entirely.

---

## Part 2: Residuals

A **residual** is the difference between an observed value and the model's prediction:

$$e_i = y_i - \hat{y}_i$$

Residuals tell us how far off the model is for each observation. Positive residuals mean the model under-predicted; negative residuals mean it over-predicted.

### 2a. Computing Residuals

Residuals are stored in the `.resid` attribute of a fitted statsmodels model. Add them to the data frame and print the first few.

In [ ]:
loan50_clean = loan50_clean.copy()
loan50_clean['fitted']   = model1.fittedvalues
loan50_clean['residual'] = model1.resid

loan50_clean[['total_income', 'loan_amount', 'fitted', 'residual']].head(10).round(1)

### 2b. Residual Plot

Plot residuals on the y-axis against fitted values on the x-axis. A well-behaved model shows residuals scattered randomly around zero with no obvious pattern.

Hint: use `ax.scatter(loan50_clean['fitted'], loan50_clean['residual'], ...)` and add `ax.axhline(0, ...)` for the zero line.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# Scatter: residuals vs. fitted values
ax.scatter(loan50_clean['fitted'], loan50_clean['residual'], alpha=0.6, edgecolors='white')

# Horizontal reference line at zero
ax.axhline(0, color='red', linestyle='--', linewidth=1)

ax.set_xlabel('Fitted values ($)')
ax.set_ylabel('Residuals ($)')
ax.set_title('Residual Plot — Loan Amount Model')
plt.tight_layout()
plt.show()

**Q3.** Do the residuals look randomly scattered, or do you notice a pattern (e.g., a fan shape, a curve)? What would a fan shape tell you about the model?

**Answer**: The residuals show a mild fan shape — the spread is larger at higher fitted values than at lower ones. This is **heteroscedasticity**: the variability in loan amounts grows with the predicted amount, which is common in financial data. A fan shape signals that the **Equal Variance (E)** condition of L-I-N-E may be violated, making standard errors and confidence intervals somewhat unreliable.

---

## Part 3: Correlation

The **correlation coefficient** $r$ measures the strength and direction of the *linear* relationship between two numerical variables. It ranges from −1 to +1:
- $r = 1$: perfect positive linear relationship
- $r = -1$: perfect negative linear relationship  
- $r = 0$: no linear relationship

Correlation is related to the regression slope:
$$b_1 = r \cdot \frac{s_y}{s_x}$$

### 3a. Computing Correlation

Compute the correlation between `total_income` and `loan_amount` two ways and verify they agree.

Hint: use `.corr()` on a DataFrame, or `np.corrcoef(x, y)[0, 1]`.

In [ ]:
# YOUR CODE HERE: compute r between total_income and loan_amount
r = loan50_clean[['total_income', 'loan_amount']].corr().iloc[0, 1]
print(f'Correlation r = {r:.4f}')

### 3b. Slope from Correlation

Verify the formula $b_1 = r \cdot s_y / s_x$ reproduces the slope from Part 1.

In [ ]:
sx = loan50_clean['total_income'].std()
sy = loan50_clean['loan_amount'].std()

# YOUR CODE HERE: compute b1 = r * sy / sx
b1_from_r = r * sy / sx

print(f'Slope from formula:  {b1_from_r:.6f}')
print(f'Slope from model:    {b1:.6f}')
print(f'Match: {np.isclose(b1_from_r, b1)}')

### 3c. Correlation vs. Causation

Compute the correlation between `poverty` and `median_hh_income` in `county`. Then make a scatterplot.

In [ ]:
county_clean = county.dropna(subset=['poverty', 'median_hh_income'])

# YOUR CODE HERE: compute correlation between poverty and median_hh_income
r_county = county_clean[['poverty', 'median_hh_income']].corr().iloc[0, 1]
print(f'r(poverty, median_hh_income) = {r_county:.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(county_clean['poverty'], county_clean['median_hh_income'],
           alpha=0.3, s=10)
ax.set_xlabel('Poverty Rate (%)')
ax.set_ylabel('Median Household Income ($)')
ax.set_title('Median Income vs. Poverty Rate (county)')
plt.tight_layout()
plt.show()

**Q4.** The correlation is negative. Does that mean higher poverty *causes* lower income? What would be a better interpretation?

**Answer**: Correlation does not imply causation. A county having a high poverty rate does not directly *cause* its median income to be low — both reflect the same underlying economic conditions (limited employment, low wages, restricted opportunity). A better interpretation: poverty rate and median household income are strongly negatively *associated* across U.S. counties. Knowing one gives useful information about the other, but observational data alone cannot establish a causal link.

---

## Part 4: The Regression Equation and Interpretation

Now fit a model to the county data: predict `median_hh_income` from `poverty`. This relationship is cleaner and the interpretation is more interesting.

In [ ]:
# Fit the model (provided)
model2 = smf.ols('median_hh_income ~ poverty', data=county_clean).fit()
print(model2.summary())

### 4a. Extracting and Interpreting Coefficients

Extract the slope and intercept, and write out the prediction equation.

In [ ]:
# YOUR CODE HERE: extract intercept and slope
# Hint: model2.params['Intercept'] and model2.params['poverty']
b0_2 = model2.params['Intercept']
b1_2 = model2.params['poverty']

print(f'median_hh_income = {b0_2:,.1f} + {b1_2:,.1f} * poverty')

**Q5.** Interpret the slope in plain English. A 1-unit increase in poverty rate is associated with how much change in median household income?

**Answer**: The slope is negative. Each additional percentage point in the poverty rate is associated with a decrease of several hundred to a few thousand dollars in median household income, on average across U.S. counties (the exact magnitude is printed above). This is a descriptive association: counties with higher poverty rates tend to have lower median incomes. As with Q4, this does not imply a simple causal relationship — both are shaped by many of the same underlying factors.

### 4b. The Regression Line Passes Through $(\bar{x}, \bar{y})$

One key property of the least squares line: it always passes through the point of means. Verify this.

In [ ]:
x_bar = county_clean['poverty'].mean()
y_bar = county_clean['median_hh_income'].mean()

# YOUR CODE HERE: plug x_bar into the equation and compare to y_bar
y_hat_at_mean = b0_2 + b1_2 * x_bar

print(f'ȳ (actual mean of y):     ${y_bar:,.1f}')
print(f'ŷ (predicted at x̄):      ${y_hat_at_mean:,.1f}')
print(f'Match: {np.isclose(y_bar, y_hat_at_mean)}')

---

## Part 5: R² — How Much Does the Model Explain?

$R^2$ (R-squared) is the fraction of the total variation in $y$ that is explained by the linear model:

$$R^2 = 1 - \frac{\text{SS}_{\text{residuals}}}{\text{SS}_{\text{total}}} = r^2$$

An $R^2$ of 0.60 means the model explains 60% of the variation in the response; the remaining 40% is unexplained.

### 5a. Computing R²

Extract R² from both models and verify that $R^2 = r^2$ for simple linear regression.

In [ ]:
# YOUR CODE HERE: extract R² from model1 and model2
# Hint: call .rsquared on each fitted model
r2_model1 = model1.rsquared
r2_model2 = model2.rsquared

print(f'Model 1 (loan_amount ~ total_income):  R² = {r2_model1:.4f}')
print(f'Model 2 (median_hh_income ~ poverty):  R² = {r2_model2:.4f}')
print()
print(f'r² for model 2: {r_county**2:.4f}  — matches R²: {np.isclose(r2_model2, r_county**2)}')

**Q6.** Which model has a higher R²? What does that tell you about which predictor does a better job explaining its response variable?

**Answer**: Model 2 (median_hh_income ~ poverty) almost certainly has a higher R² than Model 1 (loan_amount ~ total_income). Poverty rate and median household income are both county-level aggregate economic measures that move tightly together, so poverty alone explains a large fraction of county income variation. By contrast, individual loan amounts depend on many factors beyond income (credit history, loan purpose, term, lender policies), so income alone explains a smaller share. Higher R² means the predictor accounts for more of what drives the response.

### 5b. Visualizing Explained vs. Unexplained Variation

The cell below shows total variation (deviations from the mean) alongside residual variation (deviations from the line) for a small random sample. Run it and study the picture.

In [ ]:
# A small sample to make the picture readable
rng = np.random.default_rng(42)
sample = county_clean.sample(30, random_state=42)
m_s = smf.ols('median_hh_income ~ poverty', data=sample).fit()
y_mean_s = sample['median_hh_income'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, title in zip(axes, ['Total variation (deviations from ȳ)',
                              'Residual variation (deviations from line)']):
    ax.scatter(sample['poverty'], sample['median_hh_income'],
               color='steelblue', zorder=3)
    ax.set_xlabel('Poverty Rate (%)')
    ax.set_ylabel('Median HH Income ($)')
    ax.set_title(title, fontsize=10)

# Left: deviations from mean
axes[0].axhline(y_mean_s, color='red', linestyle='--', label=f'ȳ = ${y_mean_s:,.0f}')
for _, row in sample.iterrows():
    axes[0].plot([row['poverty'], row['poverty']],
                 [y_mean_s, row['median_hh_income']],
                 color='grey', linewidth=0.8)
axes[0].legend()

# Right: deviations from regression line
x_s = np.linspace(sample['poverty'].min(), sample['poverty'].max(), 100)
y_s = m_s.params['Intercept'] + m_s.params['poverty'] * x_s
axes[1].plot(x_s, y_s, color='red', linewidth=2, label='Regression line')
for _, row in sample.iterrows():
    y_fitted = m_s.params['Intercept'] + m_s.params['poverty'] * row['poverty']
    axes[1].plot([row['poverty'], row['poverty']],
                 [y_fitted, row['median_hh_income']],
                 color='grey', linewidth=0.8)
axes[1].legend()

plt.suptitle(f'R² = {m_s.rsquared:.2f}: the line explains {m_s.rsquared:.0%} of the variation',
             y=1.01)
plt.tight_layout()
plt.show()

---

## Part 6: Categorical Predictors

Regression works with categorical predictors too, as long as we convert them to **indicator variables** (also called dummy variables): a column of 0s and 1s where 1 means "in this category."

If a categorical variable has $k$ levels, we create $k - 1$ indicator variables. The omitted level becomes the **reference group** — the intercept represents its predicted value.

### 6a. Creating an Indicator Variable

In `loan50`, `homeownership` has three levels: `rent`, `mortgage`, `own`. Create an indicator variable `is_rent` that is 1 for renters and 0 for everyone else.

In [ ]:
loan50_clean['homeownership'] = loan50_clean['homeownership'].str.lower()

# YOUR CODE HERE: create is_rent indicator
# Hint: (loan50_clean['homeownership'] == 'rent').astype(int)
loan50_clean['is_rent'] = (loan50_clean['homeownership'] == 'rent').astype(int)

print(loan50_clean['is_rent'].value_counts())

### 6b. Fitting the Model

Fit a model predicting `interest_rate` from `is_rent`. Then extract and interpret the coefficients.

In [ ]:
# YOUR CODE HERE: fit interest_rate ~ is_rent using smf.ols
# Hint: same pattern as earlier models
model3 = smf.ols('interest_rate ~ is_rent', data=loan50_clean).fit()
print(model3.summary())

In [ ]:
# YOUR CODE HERE: extract the intercept and is_rent coefficient
# Hint: model3.params['Intercept'] and model3.params['is_rent']
b0_3 = model3.params['Intercept']
b1_3 = model3.params['is_rent']

print(f'Predicted interest rate for non-renters: {b0_3:.2f}%')
print(f'Predicted interest rate for renters:     {b0_3 + b1_3:.2f}%')
print(f'Difference (renters vs. non-renters):    {b1_3:.2f} percentage points')

The cell below makes a side-by-side box plot so you can see what the model is capturing.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=loan50_clean, x='homeownership',
            order=['rent', 'mortgage', 'own'],
            y='interest_rate', ax=ax)
ax.set_title('Interest Rate by Homeownership Status (loan50)')
ax.set_xlabel('Homeownership')
ax.set_ylabel('Interest Rate (%)')
plt.tight_layout()
plt.show()

**Q7.** The `is_rent` model only compares renters to non-renters. What would you need to do to also distinguish between `mortgage` and `own`? (Hint: how many indicator variables would you need?)

**Answer**: You would need $k - 1 = 2$ indicator variables — one for `mortgage` (1 if mortgage, 0 otherwise) and one for `own` (1 if own, 0 otherwise), with one level (e.g., `rent`) serving as the reference group whose predicted value is the intercept. With only `is_rent`, mortgage holders and owners are lumped together as the non-renter reference group, masking any difference between them. The boxplot above suggests those differences may be worth capturing.

---

## Part 7: Outliers

Not all outliers affect the regression line equally. The book distinguishes two kinds:

- **High leverage**: an observation with an extreme $x$ value. It *can* pull the line strongly but doesn't necessarily.
- **Influential point**: an observation that, if removed, would substantially change the slope. All influential points have high leverage, but not vice versa.

The cell below adds an artificial extreme point to the county data and refits the model to show the effect.

In [ ]:
# Add one extreme point: very high poverty, very high income (unusual combination)
outlier_row = pd.DataFrame({'poverty': [50], 'median_hh_income': [90_000]})
county_with_outlier = pd.concat([county_clean[['poverty', 'median_hh_income']],
                                  outlier_row], ignore_index=True)

model2_orig    = smf.ols('median_hh_income ~ poverty', data=county_clean).fit()
model2_outlier = smf.ols('median_hh_income ~ poverty', data=county_with_outlier).fit()

x_plot = np.linspace(0, 55, 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(county_clean['poverty'], county_clean['median_hh_income'],
           alpha=0.3, s=10, color='steelblue', label='Data')
ax.scatter([50], [90_000], color='red', s=150, zorder=5, label='Outlier added')

ax.plot(x_plot, model2_orig.params[0]    + model2_orig.params[1]    * x_plot,
        color='steelblue', linewidth=2, label=f'Original (slope={model2_orig.params[1]:,.0f})')
ax.plot(x_plot, model2_outlier.params[0] + model2_outlier.params[1] * x_plot,
        color='red',      linewidth=2, linestyle='--',
        label=f'With outlier (slope={model2_outlier.params[1]:,.0f})')

ax.set_xlabel('Poverty Rate (%)')
ax.set_ylabel('Median HH Income ($)')
ax.set_title('Effect of an Influential Point')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Original slope:     {model2_orig.params[1]:,.1f}')
print(f'Slope with outlier: {model2_outlier.params[1]:,.1f}')

**Q8.** Did adding the outlier change the slope a little or a lot? Is this outlier high leverage, influential, or both? How would you describe the difference between those two terms to someone who hadn't taken this class?

**Answer**: Adding the outlier changes the slope substantially — compare the printed slopes above. The point at (poverty=50, income=$90,000) sits far to the right with a surprisingly high income, directly counter to the strong negative trend. It is both **high leverage** (its x value is far from the center of the distribution) and **influential** (removing it would substantially change the slope). The distinction: *leverage* means being extreme on x, giving a point the *potential* to distort the fit. *Influence* means it actually *does* change the fit. A high-leverage point may not be influential if it falls on the trend line, but this outlier is extreme on x *and* far off the trend — so it is both.